# 01 — Web Scraping: Papal Documents Corpus

**Clay Harris (jbm2rt@virginia.edu) / Text as Data / 2026-05-07**

---

Acquires the corpus by scraping two complementary sources:

| Step | Description |
|------|-------------|
| 1 | Build or load the document index (papalencyclicals.net or vatican.va) |
| 2 | Download full-text documents and save raw `.txt` files |


In [1]:
import sys
sys.path.insert(0, '..')

import os
os.environ["SCRAPER_ALLOW_INSECURE_SSL"] = "true"

import importlib
import src.scraper as scraper
scraper = importlib.reload(scraper)  # always pick up latest scraper.py edits

# Notebook-level defaults to avoid symbol-order issues when running cells interactively.
discovered = []
VATICAN_LANG = "en"

# Rebind commonly used symbols from the reloaded module
scrape_index = scraper.scrape_index
scrape_documents = scraper.scrape_documents
scrape_vatican_pope_index = scraper.scrape_vatican_pope_index
discover_vatican_pope_destinations = scraper.discover_vatican_pope_destinations
merge_documents = scraper.merge_documents
save_index = scraper.save_index
save_library_csv = scraper.save_library_csv
save_pope_coverage_csv = scraper.save_pope_coverage_csv
fetch_page = scraper.fetch_page
parse_directory = scraper.parse_directory
load_dead_link_replacements = scraper.load_dead_link_replacements
apply_dead_link_replacements = scraper.apply_dead_link_replacements
recheck_languages_from_raw = scraper.recheck_languages_from_raw
INDEX_FILE = scraper.INDEX_FILE
RAW_DIR = scraper.RAW_DIR
DATA_DIR = scraper.DATA_DIR
DEAD_LINK_REPLACEMENTS_FILE = scraper.DEAD_LINK_REPLACEMENTS_FILE

import json
import pandas as pd

2026-05-07 18:16:15,003 [INFO] NumExpr defaulting to 8 threads.


In [2]:
# ── Configuration ──────────────────────────────────────────────────────
# Index source: "cached" | "papalencyclicals" | "vatican"
SCRAPE_SOURCE              = "cached"  # submission-safe default
FORCE_REBUILD_INDEX        = False     # True → ignore existing index on disk

# Vatican crawl controls (used when SCRAPE_SOURCE == "vatican")
VATICAN_LANG               = "en"
VATICAN_MAX_INDEX_PAGES    = 500
VATICAN_RUN_ALL_POPES      = True
VATICAN_POPE_LIMIT         = 0         # 0 = no limit; >0 for testing
VATICAN_SKIP_ALREADY_INDEXED = True    # resume mode: skip already-indexed popes
VATICAN_POPE_SLUG          = "francesco"
VATICAN_POPE_LABEL         = ""        # set to auto-resolve slug from label

# Vatican discovery helper
DISCOVER_POPE_LINKS        = True
REFRESH_POPE_LINKS         = False
POPE_LINKS_CACHE_FILE      = DATA_DIR / "processed" / "vatican_pope_destinations.csv"

# Step 2: document download
RESCRAPE_SHORT_DOCS        = False     # True → delete and re-scrape truncated files
SHORT_TEXT_THRESHOLD       = 3000      # chars; below this may be truncated
RUN_TEXT_SCRAPE            = False     # True → hit the network
MAX_DOCS                   = 0         # 0 = no limit


In [3]:
# Check optional dependencies
import importlib.util

def _check_dep(package, install_cmd):
    if importlib.util.find_spec(package) is None:
        print(f"'{package}' is NOT installed — some features will be disabled.")
        print(f"   To enable: {install_cmd}")
    else:
        print(f"'{package}' is installed.")

_check_dep("lxml",     "pip install lxml")
_check_dep("ebooklib", "pip install ebooklib")


'lxml' is installed.
'ebooklib' is NOT installed — some features will be disabled.
   To enable: pip install ebooklib


## Step 1: Build / Load the Document Index

Choose a source in the next cell:
- `papalencyclicals`: parse the legacy directory page
- `vatican`: crawl a pope section (document types, optional year pages, then leaf documents)

This step prepares the records used by the download step.

In [4]:
# Optional helper: discover pope destinations automatically from Vatican source page.
# This avoids manual slug copy/paste and caches results for reuse in later cells.

if DISCOVER_POPE_LINKS:
    if ('discovered' in globals()) and isinstance(discovered, list) and discovered and not REFRESH_POPE_LINKS:
        pope_links = discovered
        print(f"Using pope links already loaded in memory: {len(pope_links)}")
    elif POPE_LINKS_CACHE_FILE.exists() and not REFRESH_POPE_LINKS:
        df_popes = pd.read_csv(POPE_LINKS_CACHE_FILE).fillna('')
        pope_links = df_popes.to_dict(orient='records')
        discovered = pope_links
        print(f"Loaded pope links from cache: {len(df_popes)}")
    else:
        pope_links = discover_vatican_pope_destinations(
            source_url="https://www.vatican.va/content/vatican/en.html",
            lang=VATICAN_LANG if 'VATICAN_LANG' in globals() else 'en',
        )
        discovered = pope_links
        df_popes = pd.DataFrame(pope_links)
        POPE_LINKS_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        df_popes.to_csv(POPE_LINKS_CACHE_FILE, index=False, encoding='utf-8')
        print(f"Discovered pope profile links and cached to: {POPE_LINKS_CACHE_FILE}")

    df_popes = pd.DataFrame(pope_links)
    print(f"Total pope profile links available: {len(df_popes)}")
    if len(df_popes):
        print(df_popes.head(25).to_string(index=False))

        # Suggested modern-pope choices for detailed Vatican crawling.
        modern = df_popes[df_popes['pope_slug'].isin(['leo-xiv', 'francesco', 'benedict-xvi', 'john-paul-ii'])]
        if len(modern):
            print("\nSuggested modern pope slugs:")
            print(modern[['pope_label', 'pope_slug', 'pope_home_url']].to_string(index=False))

Loaded pope links from cache: 252
Total pope profile links available: 252
                                 pope_label                                                                pope_card_url      pope_slug                                         pope_home_url
               Leo XIV 8,18.V.2025 --/--/--         https://www.vatican.va/content/vatican/en/holy-father/leone-xiv.html        leo-xiv        https://www.vatican.va/content/leo-xiv/en.html
          Francis 13,19.III.2013 21.IV.2025         https://www.vatican.va/content/vatican/en/holy-father/francesco.html      francesco      https://www.vatican.va/content/francesco/en.html
      Benedict XVI 19,24.IV.2005 28.II.2013     https://www.vatican.va/content/vatican/en/holy-father/benedetto-xvi.html   benedict-xvi    http://www.vatican.va/content/benedict-xvi/en.html
        John Paul II 16,22.X.1978 2.IV.2005 https://www.vatican.va/content/vatican/en/holy-father/giovanni-paolo-ii.html   john-paul-ii    http://www.vatican.va/conte

In [5]:
%%time
# Example labels: "Benedict XVI", "Francis", "John Paul II", "John XXIII"


existing_documents = []
if INDEX_FILE.exists() and not FORCE_REBUILD_INDEX:
    with open(INDEX_FILE, "r", encoding="utf-8") as f:
        existing_documents = json.load(f)
    print(f"Loaded existing index: {len(existing_documents)} documents")
elif FORCE_REBUILD_INDEX:
    print("FORCE_REBUILD_INDEX=True — ignoring existing index on disk")

if SCRAPE_SOURCE == "cached":
    if not existing_documents:
        raise ValueError("SCRAPE_SOURCE='cached' requires an existing index file.")
    documents = existing_documents
    documents_for_scrape = []
    print("Cached mode active: no index crawling will be performed.")

elif SCRAPE_SOURCE == "papalencyclicals":
    if existing_documents and not FORCE_REBUILD_INDEX:
        documents = existing_documents
        print(f"Using existing papalencyclicals index: {len(documents)} documents")
    else:
        documents = scrape_index()
        save_index(documents)
        print(f"Scraped papalencyclicals index: {len(documents)} documents")

    # The scrape step will process this collection directly.
    documents_for_scrape = documents

elif SCRAPE_SOURCE == "vatican":
    documents = existing_documents
    documents_for_scrape = []

    # Reuse discovery results from step 1 when available to avoid repeat network work.
    if ('discovered' in globals()) and isinstance(discovered, list) and discovered:
        pope_lookup = discovered
        print(f"Using in-memory pope lookup with {len(pope_lookup)} rows")
    else:
        pope_lookup = discover_vatican_pope_destinations(
            source_url="https://www.vatican.va/content/vatican/en.html",
            lang=VATICAN_LANG,
        )
        discovered = pope_lookup
        print(f"Built pope lookup on demand with {len(pope_lookup)} rows")

    if VATICAN_RUN_ALL_POPES:
        ordered_slugs = []
        slug_to_label = {}
        slug_to_start_url = {}
        for row in pope_lookup:
            slug = str(row.get('pope_slug', '')).strip().lower()
            if not slug:
                continue
            if slug not in slug_to_label:
                ordered_slugs.append(slug)
                slug_to_label[slug] = row.get('pope_label', '')

            # IMPORTANT: Prefer pope card URL (/content/vatican/.../holy-father/*.html),
            # which is the reliable entry point for older popes.
            card_url = str(row.get('pope_card_url', '')).strip()
            home_url = str(row.get('pope_home_url', '')).strip()
            start_url = card_url or home_url
            if start_url and slug not in slug_to_start_url:
                slug_to_start_url[slug] = start_url

        indexed_slug_lang = set()
        for doc in existing_documents:
            doc_id = str(doc.get('doc_id', ''))
            if not doc_id.startswith('vatican__'):
                continue
            parts = doc_id.split('__', 3)
            if len(parts) < 4:
                continue
            indexed_slug_lang.add((parts[1].lower(), parts[2].lower()))

        if VATICAN_SKIP_ALREADY_INDEXED:
            before = len(ordered_slugs)
            ordered_slugs = [
                s for s in ordered_slugs
                if (s.lower(), VATICAN_LANG.lower()) not in indexed_slug_lang
            ]
            print(f"Resume mode: skipping {before - len(ordered_slugs)} already-indexed pope slug(s)")

        if VATICAN_POPE_LIMIT > 0:
            ordered_slugs = ordered_slugs[:VATICAN_POPE_LIMIT]

        print(f"Running Vatican index crawl for {len(ordered_slugs)} pope slug(s)")
        failed_popes = []

        for i, slug in enumerate(ordered_slugs, start=1):
            label = slug_to_label.get(slug, '')
            label_suffix = f" ({label})" if label else ""
            start_url = slug_to_start_url.get(slug, '')
            start_hint = f" | start={start_url}" if start_url else ""
            print(f"[{i}/{len(ordered_slugs)}] Crawling {slug}{label_suffix}{start_hint}")
            try:
                pope_docs = scrape_vatican_pope_index(
                    pope_slug=slug,
                    lang=VATICAN_LANG,
                    start_url=start_url,
                    max_index_pages=VATICAN_MAX_INDEX_PAGES,
                )
                print(f"  -> discovered {len(pope_docs)} documents")
                documents_for_scrape.extend(pope_docs)
                documents = merge_documents(documents, pope_docs)
                save_index(documents)  # checkpoint after each pope
                indexed_slug_lang.add((slug.lower(), VATICAN_LANG.lower()))
                print(f"  -> merged index size now {len(documents)}")
            except KeyboardInterrupt:
                print("Crawl interrupted by user. Partial progress has been saved.")
                raise
            except Exception as ex:
                failed_popes.append((slug, str(ex)))
                print(f"  -> failed: {ex}")

        print(
            f"Batch crawl complete: {len(documents_for_scrape)} docs discovered across "
            f"{len(ordered_slugs)} pope slug(s)"
        )
        if failed_popes:
            print(f"Failed pope slugs: {len(failed_popes)}")
            for slug, err in failed_popes[:20]:
                print(f" - {slug}: {err}")
            if len(failed_popes) > 20:
                print(f" ... and {len(failed_popes) - 20} more")

    else:
        if VATICAN_POPE_LABEL.strip():
            matches = [
                r for r in pope_lookup
                if VATICAN_POPE_LABEL.strip().lower() in (r.get('pope_label', '').lower())
            ]
            if not matches:
                raise ValueError(
                    f"No pope label match for '{VATICAN_POPE_LABEL}'. "
                    "Run the discovery helper cell to see available labels."
                )
            VATICAN_POPE_SLUG = matches[0]['pope_slug']
            print(f"Resolved pope label '{VATICAN_POPE_LABEL}' -> slug '{VATICAN_POPE_SLUG}'")

        vatican_documents = scrape_vatican_pope_index(
            pope_slug=VATICAN_POPE_SLUG,
            lang=VATICAN_LANG,
            max_index_pages=VATICAN_MAX_INDEX_PAGES,
        )
        print(
            f"Crawled Vatican index for {VATICAN_POPE_SLUG}/{VATICAN_LANG}: "
            f"{len(vatican_documents)} documents discovered"
        )

        documents_for_scrape = vatican_documents
        documents = merge_documents(existing_documents, vatican_documents)
        save_index(documents)
        print(f"Merged index size after Vatican crawl: {len(documents)} documents")

else:
    raise ValueError("SCRAPE_SOURCE must be 'cached', 'papalencyclicals' or 'vatican'")

Loaded existing index: 17436 documents
Cached mode active: no index crawling will be performed.
CPU times: user 49.4 ms, sys: 10.4 ms, total: 59.7 ms
Wall time: 61.9 ms


In [6]:
print(f'Index entries:  {len(documents):,}')
print(f'Unique doc_ids: {len({d["doc_id"] for d in documents}):,}')
pd.DataFrame(documents).head()


Index entries:  17,436
Unique doc_ids: 17,430


,doc_id,category,author,author_dates,pope,pope_dates,title,url,year,document_type,language,format,text_length,source,collection,detail_level,error,original_url,pope_slug
0,pope_alexander_iv__clara_claris_praeclara,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Clara claris praeclara,http://www.franciscan-archive.org/bullarium/cl...,1255,document,en,html,35694,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
1,pope_alexander_iv__dignum_arbitramur_et_congruum,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Dignum arbitramur et congruum,https://www.papalencyclicals.net//alex04/alex4...,1255,encyclical,la,html,1908,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
2,pope_alexander_iv__petitionibus_vestris_benign...,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Petitionibus vestris benignum impertientes,https://www.papalencyclicals.net//alex04/alex4...,1255,encyclical,en,html,946,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
3,pope_alexander_iv__inter_ea_quae_placita,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Inter ea quae placita,https://www.papalencyclicals.net//alex04/alex4...,1255,encyclical,la,html,2287,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
4,pope_alexander_vi__inter_caetera,pope,Pope Alexander VI,1492-1503,Pope Alexander VI,1492-1503,Inter Caetera,https://www.papalencyclicals.net//alex06/alex0...,1493,document,en,html,10046,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN


In [7]:
# Preview the index
import numpy as np

df_index = pd.DataFrame(documents)

# Backfill normalized fields for legacy index rows loaded from disk.
if 'category' not in df_index.columns:
    df_index['category'] = np.where(
        df_index.get('pope', '').fillna('').str.lower().eq('church councils'),
        'council',
        'pope'
    )
if 'author' not in df_index.columns:
    df_index['author'] = np.where(
        df_index['category'].eq('council'),
        df_index.get('title', ''),
        df_index.get('pope', '')
    )
if 'author_dates' not in df_index.columns:
    df_index['author_dates'] = np.where(
        df_index['category'].eq('council'),
        '',
        df_index.get('pope_dates', '')
    )

print('Documents by category:')
print(df_index['category'].value_counts())

print('\nTop authors:')
print(df_index['author'].value_counts().head(20))

# Years can be empty strings or malformed; coerce to numeric before range stats.
years = pd.to_numeric(df_index['year'], errors='coerce')
if years.notna().any():
    print(f"\nYears covered (valid only): {int(years.min())} - {int(years.max())}")
else:
    print('\nYears covered (valid only): n/a')

print(f"Missing/invalid year values: {years.isna().sum()} of {len(df_index)}")

Documents by category:
category
pope       17415
council       21
Name: count, dtype: int64

Top authors:
author
John Paul Ii             6155
Francesco                5747
Benedict Xvi             3125
Paul Vi                   877
Leo Xiv                   545
Pius Xii                  220
Pope Leo XIII              88
Leo Xiii                   87
Pope St. John Paul II      60
Pope Pius XII              60
Pope Benedict XIV          44
Pope Bl. Pius IX           42
Pope Paul VI               36
John Xxiii                 33
Pope Pius XI               32
Pope Pius VI               26
Pope St. Pius X            26
Pius Xi                    25
John Paul I                22
Pope Benedict XVI          21
Name: count, dtype: int64

Years covered (valid only): 1215 - 2016
Missing/invalid year values: 16908 of 17436


In [8]:
# Audit source domains to troubleshoot external links (e.g., digilander)
from urllib.parse import urlparse

df_domains = df_index.copy()
df_domains['domain'] = df_domains['url'].fillna('').apply(lambda u: urlparse(u).netloc.lower())
print('Top source domains:')
print(df_domains['domain'].value_counts().head(20))

external = df_domains[~df_domains['domain'].str.contains('papalencyclicals.net|vatican.va', regex=True, na=False)]
print(f"\nExternal-domain records: {len(external)}")
if len(external):
    print(external[['pope', 'title', 'url']].head(25).to_string(index=False))

Top source domains:
domain
www.vatican.va                16990
www.papalencyclicals.net        402
w2.vatican.va                    27
web.archive.org                   7
www.franciscan-archive.org        5
www.bluewaterarts.com             1
en.wikisource.org                 1
www.ewtn.com                      1
www.nativeweb.org                 1
www.adoremus.org                  1
Name: count, dtype: int64

External-domain records: 17
                 pope                                                          title                                                                                                                                                                             url
    Pope Alexander IV                                         Clara claris praeclara                                                                                                                          http://www.franciscan-archive.org/bullarium/clara.html
      Pope Clement XI               

In [9]:
# Review curated dead-link replacements and identify any remaining gaps
dead_domains = set(scraper.KNOWN_DEAD_DOMAINS)

dead_links = df_domains[df_domains['domain'].isin(dead_domains)].copy()
print(f"Dead-link records in index: {len(dead_links)}")

if DEAD_LINK_REPLACEMENTS_FILE.exists():
    df_replacements = pd.read_csv(DEAD_LINK_REPLACEMENTS_FILE).fillna('')
    usable = df_replacements[df_replacements['replacement_url'].str.strip().ne('')].copy()
    missing = df_replacements[df_replacements['replacement_url'].str.strip().eq('')].copy()

    print(f"Curated replacement rows: {len(df_replacements)}")
    print(f"Usable replacements: {len(usable)}")
    print(f"Still missing replacement URLs: {len(missing)}")

    preview_cols = ['doc_id', 'url', 'replacement_url']
    if len(usable):
        print("\nSample replacement mappings:")
        print(usable[preview_cols].head(15).to_string(index=False))

    if len(missing):
        print("\nRows still missing replacement URLs:")
        print(missing[preview_cols].head(15).to_string(index=False))
else:
    print(f"No curated replacement file found at {DEAD_LINK_REPLACEMENTS_FILE}")
    print("Run the domain audit above and create the CSV before scraping dead links.")

Dead-link records in index: 0
Curated replacement rows: 33
Usable replacements: 33
Still missing replacement URLs: 0

Sample replacement mappings:
                                   doc_id                                                url                                                                                                                  replacement_url
 pope_clement_xiii__accedamus_cum_fiducia    http://digilander.iol.it/magistero/c13acced.htm                         https://www.vatican.va/content/clemens-xiii/it/documents/breve-accedamus-cum-fiducia-25-giugno-1768.html
    pope_clement_xiii__pastoralis_officii    http://digilander.iol.it/magistero/c13pasto.htm                                                                                https://www.papalencyclicals.net/leo13/l13dul.htm
         pope_clement_xiii__quam_graviter    http://digilander.iol.it/magistero/c13quamg.htm                             https://www.vatican.va/content/clemens-xiii/it/documents/enciclica

In [10]:
# Apply curated replacements before scraping so the preview matches scraper behavior
replacement_map = load_dead_link_replacements()
documents, replaced_doc_ids = apply_dead_link_replacements(documents, replacement_map)

print(f"Documents using replacement URLs this run: {len(replaced_doc_ids)}")
if replaced_doc_ids:
    df_replaced = pd.DataFrame([
        {
            'doc_id': doc['doc_id'],
            'original_url': doc.get('original_url', ''),
            'url': doc.get('url', ''),
        }
        for doc in documents
        if doc['doc_id'] in replaced_doc_ids
    ])
    print(df_replaced.head(15).to_string(index=False))

2026-05-07 18:16:15,523 [INFO] Loaded 33 dead-link replacements from /Users/queclay/Documents/MSDS/DS5001/encyclicals/data/processed/dead_link_replacements.csv


Documents using replacement URLs this run: 0


In [11]:
# Optional finalization step: recheck papalencyclicals language labels from local raw files only.
# This does NOT download or scrape any pages.
RECHECK_PAPAL_LANGUAGES = True
RECHECK_ONLY_CURRENT_EN = True
RECHECK_MIN_TEXT_CHARS = 500

if RECHECK_PAPAL_LANGUAGES:
    report = recheck_languages_from_raw(
        documents,
        source="papalencyclicals.net",
        only_if_current="en" if RECHECK_ONLY_CURRENT_EN else None,
        min_text_chars=RECHECK_MIN_TEXT_CHARS,
    )
    print(
        "Language recheck summary: "
        f"examined={report['examined']} updated={report['updated']} "
        f"missing_raw={report['skipped_missing_raw']} short={report['skipped_short']}"
    )

    if report["updated"]:
        df_lang_changes = pd.DataFrame(report["changes"])
        print("\nUpdated rows by new language:")
        print(df_lang_changes["new_language"].value_counts())
        print("\nSample language updates:")
        print(df_lang_changes[["doc_id", "old_language", "new_language", "title"]].head(30).to_string(index=False))

    # Persist metadata updates so subsequent runs are stable and offline-friendly.
    save_index(documents)
    save_library_csv(documents)
    save_pope_coverage_csv(documents)
else:
    print("RECHECK_PAPAL_LANGUAGES=False — skipping language recheck.")

2026-05-07 18:16:16,060 [INFO] Saved index with 17436 documents to /Users/queclay/Documents/MSDS/DS5001/encyclicals/data/encyclicals_index.json


Language recheck summary: examined=552 updated=66 missing_raw=1 short=6

Updated rows by new language:
new_language
it    59
la     6
fr     1
Name: count, dtype: int64

Sample language updates:
                                                   doc_id old_language new_language                                 title
pope_alexander_vii__super_cathedram_principis_apostolorum           en           la Super Cathedram Principis Apostolorum
                pope_benedict_xiv__accepimus_praestantium           en           it                Accepimus praestantium
                       pope_benedict_xiv__benedictus_deus           en           it                       Benedictus Deus
                   pope_benedict_xiv__celebrationem_magni           en           it                   Celebrationem magni
                     pope_benedict_xiv__certiores_effecti           en           it                     Certiores effecti
                      pope_benedict_xiv__cum_illud_semper           en   

2026-05-07 18:16:16,160 [INFO] Saved LIBRARY.csv with 17436 rows
2026-05-07 18:16:16,170 [INFO] Saved pope coverage summary to /Users/queclay/Documents/MSDS/DS5001/encyclicals/data/processed/POPE_COVERAGE.csv


## Step 2: Download Document Texts

Submission-safe default: keep `RUN_TEXT_SCRAPE = False` in the next cell so this notebook does not hit the network or re-scrape already collected documents.

If you explicitly need a refresh, set `RUN_TEXT_SCRAPE = True` and optionally cap with `MAX_DOCS`.

In [12]:
# Identify papalencyclicals.net documents with suspiciously short text that
# may have been truncated by the old extraction logic (text in raw text nodes
# rather than <p> tags). Those raw files are deleted here so the scrape step
# below re-downloads them with the improved extractor.
#
# Set RESCRAPE_SHORT_DOCS = False to skip this step and keep cached files.

df_scrape = pd.DataFrame(documents)
suspect_mask = (
    df_scrape['url'].fillna('').str.contains('papalencyclicals.net', regex=False)
    & df_scrape['text_length'].gt(0)
    & df_scrape['text_length'].lt(SHORT_TEXT_THRESHOLD)
    & df_scrape['language'].ne('unknown')
)
suspect_docs = df_scrape[suspect_mask].copy()
print(f"Potentially truncated papalencyclicals.net docs (text_length < {SHORT_TEXT_THRESHOLD}): {len(suspect_docs)}")
if len(suspect_docs):
    print(suspect_docs[['doc_id', 'text_length', 'url']].sort_values('text_length').to_string(index=False))

if RESCRAPE_SHORT_DOCS and len(suspect_docs):
    deleted = []
    for doc_id in suspect_docs['doc_id']:
        raw = RAW_DIR / f"{doc_id}.txt"
        if raw.exists():
            raw.unlink()
            deleted.append(doc_id)
    print(f"\nCleared {len(deleted)} cached files — they will be re-scraped below.")
elif not RESCRAPE_SHORT_DOCS:
    print("\nRESCRAPE_SHORT_DOCS=False — skipping cache clear.")

Potentially truncated papalencyclicals.net docs (text_length < 3000): 18
                                                       doc_id  text_length                                                                          url
                                 pope_st__pius_x__septimo_iam          400                     https://www.papalencyclicals.net//pius10/septimo-iam.htm
                                pope_leo_xii__quanta_laetitia          400                  https://www.papalencyclicals.net//leo12/quanta-laetitia.htm
                                 pope_pius_xi__rerum_condicio          400                  https://www.papalencyclicals.net//pius11/rerum-condicio.htm
                 pope_bl__pius_ix__dives_in_misericordia_deus          470 https://www.papalencyclicals.net/wp-content/uploads/2026/01/PiusIX-Dives.pdf
pope_alexander_iv__petitionibus_vestris_benignum_impertientes          946                      https://www.papalencyclicals.net//alex04/alex4petit.htm
               

In [13]:
%%time
# Scrape document texts only when explicitly enabled.
# Default is submission-safe and offline-friendly.


if RUN_TEXT_SCRAPE:
    # In Vatican mode documents_for_scrape only holds newly discovered docs, so we
    # pass the complete index to ensure every undownloaded document is fetched.
    scrape_target = documents if SCRAPE_SOURCE == "vatican" else documents_for_scrape

    scraped_documents = scrape_documents(
        scrape_target,
        max_docs=MAX_DOCS,
        show_progress=True,
        progress_label="Downloading"
    )

    # In Vatican mode scrape_documents mutates the records in-place, so documents
    # already reflects the updated text_length/language/year fields.
    # For papalencyclicals mode the returned list IS the full document set.
    if SCRAPE_SOURCE != "vatican":
        documents = scraped_documents

    print("Text scraping run completed.")
else:
    print("RUN_TEXT_SCRAPE=False — skipping text download/scrape step.")

# Always persist current metadata snapshot for downstream notebooks.
save_index(documents)
save_library_csv(documents)
save_pope_coverage_csv(documents)
df.head()


RUN_TEXT_SCRAPE=False — skipping text download/scrape step.


2026-05-07 18:16:16,427 [INFO] Saved index with 17436 documents to /Users/queclay/Documents/MSDS/DS5001/encyclicals/data/encyclicals_index.json
2026-05-07 18:16:16,529 [INFO] Saved LIBRARY.csv with 17436 rows
2026-05-07 18:16:16,539 [INFO] Saved pope coverage summary to /Users/queclay/Documents/MSDS/DS5001/encyclicals/data/processed/POPE_COVERAGE.csv


NameError: name 'df' is not defined

In [14]:
# Summary
df = pd.DataFrame(documents)
print(f"Total documents: {len(df)}")
print(f"\nBy language:")
print(df['language'].value_counts())
print(f"\nBy format:")
print(df['format'].value_counts())
print(f"\nDocuments with text: {(df['text_length'] > 100).sum()}")
df.head()


Total documents: 17436

By language:
language
en         17359
it            59
la            10
unknown        5
               2
fr             1
Name: count, dtype: int64

By format:
format
text     16866
html       566
error        4
Name: count, dtype: int64

Documents with text: 17432


,doc_id,category,author,author_dates,pope,pope_dates,title,url,year,document_type,language,format,text_length,source,collection,detail_level,error,original_url,pope_slug
0,pope_alexander_iv__clara_claris_praeclara,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Clara claris praeclara,http://www.franciscan-archive.org/bullarium/cl...,1255,document,en,html,35694,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
1,pope_alexander_iv__dignum_arbitramur_et_congruum,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Dignum arbitramur et congruum,https://www.papalencyclicals.net//alex04/alex4...,1255,encyclical,la,html,1908,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
2,pope_alexander_iv__petitionibus_vestris_benign...,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Petitionibus vestris benignum impertientes,https://www.papalencyclicals.net//alex04/alex4...,1255,encyclical,en,html,946,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
3,pope_alexander_iv__inter_ea_quae_placita,pope,Pope Alexander IV,1254-1261,Pope Alexander IV,1254-1261,Inter ea quae placita,https://www.papalencyclicals.net//alex04/alex4...,1255,encyclical,la,html,2287,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN
4,pope_alexander_vi__inter_caetera,pope,Pope Alexander VI,1492-1503,Pope Alexander VI,1492-1503,Inter Caetera,https://www.papalencyclicals.net//alex06/alex0...,1493,document,en,html,10046,papalencyclicals.net,legacy_encyclicals,standard,NaN,NaN,NaN


In [15]:
# List raw text files
raw_files = list(RAW_DIR.glob('*.txt'))
print(f"Raw text files on disk: {len(raw_files)}")
sizes = [f.stat().st_size for f in raw_files]
print(f"Total size: {sum(sizes)/1024/1024:.1f} MB")
print(f"Average size: {sum(sizes)/len(sizes)/1024:.1f} KB")

Raw text files on disk: 17466
Total size: 139.8 MB
Average size: 8.2 KB
